# 06 — Seed export

Collapses the per-domain calibration tables (`tac/data/`,
`electricity_pricing/data/`, `facility_calibration/data/`,
`route_context/data/`, plus the shared `data/sources_register.csv`)
into the seed CSVs that
`db/dev/seed.py` reads, written to `calib/seed/`:

- `sources.csv` — the full source register, seeded as `input_params.sources`
  rows (keyed by `source_key` prefix in the description).
- `track_tac.csv` — DB-ready TAC columns per country, **EUR-converted**
  (this is the single place FX happens; `calc_tac.py` and the DB never see
  native currency): the flat indicative `track_tac_eur_train_km` display
  value for NT-REF plus the full component group
  (`track_tac_b_day` … `track_tac_peak_weekdays_only`).
- `passage_charges.csv` — crossing charges (EUR) with the crossing polygon
  as a GeoJSON string per row.
- `track_infrastructures.csv` / `track_infrastructure_defaults.csv` — the
  wider infra columns (parking, shunting, energy, terrain, buffer). Of
  these, seed.py currently consumes only the TAC and source files — the
  other infra parameters stay hardcoded in seed.py until their models are
  implemented.

**Every cell in this notebook is stdlib-only.** `db/dev/seed.py` regenerates
missing seed CSVs by exec'ing the pandas-free cells of a calibration notebook,
and the API container has no dev extras — so a stray `import pandas` here
breaks seeding in production. There is no local-module import either, for the
same reason: `resolution.py` and friends are not on the path under that exec.

Seed CSVs are **derived artifacts and are not committed**. `data/` is the
committed truth; re-run `02`–`05` then this notebook to regenerate.


In [ ]:
# STDLIB-ONLY cell. See the seed-export contract in calib/README.md.
import csv
from pathlib import Path


def _calib_dir() -> Path:
    """Anchor on the calib folder whatever the kernel's cwd happens to be."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "resolution.py").exists():
            return cand
    for sub in ("backend/models/infrastructure/calib", "models/infrastructure/calib"):
        if (cwd / sub).is_dir():
            return (cwd / sub).resolve()
    raise RuntimeError("cannot locate models/infrastructure/calib")


CALIB_DIR = _calib_dir()
DATA_DIR = CALIB_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)


def write_data(name: str, fieldnames: list[str], rows: list[dict]) -> None:
    """Write one committed observation table to calib/data/."""
    with open(DATA_DIR / name, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"data/{name}: {len(rows)} rows")


def read_data(name: str) -> list[dict]:
    with open(DATA_DIR / name, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_domain_data(domain: str, name: str) -> list[dict]:
    """Read a committed observation table from one calibration domain's
    own data/ folder (calib/<domain>/data/<name>) — tac,
    electricity_pricing, facility_calibration, route_context. Domain-local
    data stays domain-local; only sources_register.csv (via read_data) is
    shared at the calib/ root.
    """
    path = CALIB_DIR / domain / "data" / name
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


SEED_DIR = CALIB_DIR / "seed"
SEED_DIR.mkdir(exist_ok=True)


def write_seed(name: str, fieldnames: list[str], rows: list[dict]) -> None:
    with open(SEED_DIR / name, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"seed/{name}: {len(rows)} rows")

## Sources

`input_params.sources` uses a SERIAL primary key, so the seed carries a stable
`source_key` (our semantic id) and `seed.py` resolves it to the generated
integer when it inserts. That keeps the calibration free of database ids while
still letting every parameter point at its document.

In [ ]:
register = read_data("sources_register.csv")
SOURCE_FIELDS = ["source_key", "source_description", "source_url", "source_date"]
src_rows = [
    {
        "source_key": r["source_id"],
        "source_description": f"{r['title']} — {r['publisher']} ({r['pub_year']})",
        "source_url": r["url_or_file"] if r["url_or_file"].startswith("http") else "",
        "source_date": r["date_accessed"],
    }
    for r in register
]
write_seed("sources.csv", SOURCE_FIELDS, src_rows)

## track_infrastructures

One row per country. The `_src` columns carry the `source_key` of the value
that drove each field, so provenance survives into the database rather than
stopping at the notebook.

Two shape mismatches against the current schema are handled here and flagged
for the migration discussion rather than silently papered over:

- **TAC** is a single `NUMERIC(8,2)` column, but the component model needs
  `b_day`, `b_night`, `gamma` and the rest. This notebook evaluates the
  components for the reference night train and writes the resulting flat
  EUR/train-km figure, so the current schema seeds correctly today. The
  component detail is written alongside to `seed/track_tac_components.csv`,
  ready for the extended table.
- **Terrain category** is constrained to Flat/Hilly/Mountainous, so the five
  model bands are mapped down. The unmapped band travels in
  `track_tac_components.csv` too.

In [ ]:
FX_TO_EUR = {
    "EUR": 1.0,
    "CHF": 1.064,
    "HUF": 1 / 396.0,
    "SEK": 1 / 11.2,
    "GBP": 1.20,
    "DKK": 1 / 7.46,
    "NOK": 1 / 11.7,
    "PLN": 1 / 4.25,
    "CZK": 1 / 24.5,
    "RON": 1 / 5.08,
}
NT_GROSS_T, NT_TRAIN_KM = 600.0, 1.0  # evaluate per train-km


def _idx(rows, key="country_code"):
    out: dict = {}
    for r in rows:
        out.setdefault(r[key], {})[r["parameter"]] = r
    return out


tac = _idx(read_domain_data("tac", "tac_components.csv"))
elec = {
    r["country_code"]: r
    for r in read_domain_data("electricity_pricing", "electricity_price_modes.csv")
}
fac = {
    r["country_code"]: r
    for r in read_domain_data("facility_calibration", "facility_reference_rotation.csv")
}
ctx = {
    r["country_code"]: r
    for r in read_domain_data("route_context", "route_context_summary.csv")
}
facv = _idx(read_domain_data("facility_calibration", "facility_charges.csv"))
elecv = _idx(read_domain_data("electricity_pricing", "electricity_prices.csv"))
ctxv = _idx(read_domain_data("route_context", "route_context.csv"))


def _f(row: dict | None):
    """Value as float, or None when the parameter is missing/no_railway."""
    if not row or row["status"] in ("missing", "no_railway") or row["value"] == "":
        return None
    return float(row["value"])


def tac_per_train_km(cc: str):
    """Evaluate the component model for NT-REF, in EUR per train-km."""
    p = tac.get(cc, {})
    night = _f(p.get("b_night"))
    day = _f(p.get("b_day"))
    base = night if night is not None else day
    gamma = _f(p.get("gamma"))
    if base is None and gamma is None:
        return None, ""
    src_row = p.get("b_night") if night is not None else p.get("b_day")
    if src_row is None or not src_row.get("source_id"):
        src_row = p.get("gamma") or {}
    ccy = (src_row.get("currency") or "EUR") if src_row else "EUR"
    total = (base or 0.0) + (gamma or 0.0) * NT_GROSS_T
    return total * FX_TO_EUR.get(ccy, 1.0), src_row.get("source_id", "")


def _f_eur(row: dict | None):
    """Component value converted to EUR using the row's own currency —
    each SV carries its native currency, so conversion is per component,
    not per country."""
    v = _f(row)
    if v is None:
        return None
    return v * FX_TO_EUR.get(row.get("currency") or "EUR", 1.0)

In [ ]:
TI_FIELDS = [
    "country_code",
    "track_tac_eur_train_km",
    "track_tac_src",
    "track_parking_eur_day",
    "track_parking_src",
    "track_shunting_eur_event",
    "track_shunting_src",
    "track_energy_price_eur_kwh",
    "track_energy_price_src",
    "track_terrain_category",
    "track_terrain_score",
    "track_terrain_src",
    "track_hsr_allowed",
    "track_hsr_src",
    "track_min_boarding_time",
    "track_min_boarding_src",
    "track_min_alighting_time",
    "track_min_alighting_src",
    "track_buffer_quota_per",
    "track_buffer_src",
    "change_log",
]

# DB-ready TAC columns (input_params.track_infrastructures names), all EUR
# — this file is what seed.py merges onto its canonical country rows.
TAC_FIELDS = [
    "country_code",
    "track_tac_eur_train_km",
    "track_tac_src",
    "track_tac_night_mode",
    "track_tac_night_band_start",
    "track_tac_night_band_end",
    "track_tac_night_full_if_accommodation",
    "track_tac_b_day",
    "track_tac_b_night",
    "track_tac_gamma",
    "track_tac_seat_km",
    "track_tac_revenue_share",
    "track_tac_fixed_per_train_km",
    "track_tac_per_stop",
    "track_tac_peak_multiplier",
    "track_tac_congestion_surcharge_eur_km",
    "track_tac_peak_band1_start",
    "track_tac_peak_band1_end",
    "track_tac_peak_band2_start",
    "track_tac_peak_band2_end",
    "track_tac_peak_weekdays_only",
]

modes = {r["country_code"]: r for r in read_domain_data("tac", "tac_night_mode.csv")}
peaks = {r["country_code"]: r for r in read_domain_data("tac", "tac_peak_bands.csv")}
ti_rows, comp_rows = [], []

for cc in sorted(ctx):
    tac_v, tac_src = tac_per_train_km(cc)
    f, e, c = fac.get(cc, {}), elec.get(cc, {}), ctx[cc]
    park_src = (facv.get(cc, {}).get("parking_per_event_ref") or {}).get(
        "source_id", ""
    )
    shunt_src = (facv.get(cc, {}).get("shunting_allin") or {}).get("source_id", "")
    elec_src = (elecv.get(cc, {}).get("working_price") or {}).get("source_id", "")
    ctx_src = "RMMS-9"

    ti_rows.append(
        {
            "country_code": cc,
            "track_tac_eur_train_km": "" if tac_v is None else round(tac_v, 2),
            "track_tac_src": tac_src,
            "track_parking_eur_day": f.get("parking_eur_event_ref", ""),
            "track_parking_src": park_src,
            "track_shunting_eur_event": f.get("shunting_allin_eur_event", ""),
            "track_shunting_src": shunt_src,
            "track_energy_price_eur_kwh": e.get("working_price_eur_kwh", ""),
            "track_energy_price_src": elec_src,
            "track_terrain_category": c["terrain_category_db"],
            "track_terrain_score": c["terrain_score"],
            "track_terrain_src": "",
            "track_hsr_allowed": c["hsr_allowed"],
            "track_hsr_src": "",
            "track_min_boarding_time": f"00:{int(float(c['min_dwell_min'])):02d}:00",
            "track_min_boarding_src": "",
            "track_min_alighting_time": f"00:{int(float(c['min_dwell_min'])):02d}:00",
            "track_min_alighting_src": "",
            "track_buffer_quota_per": round(
                float(c["timetable_buffer_pct"]) / 100.0, 4
            ),
            "track_buffer_src": ctx_src,
            "change_log": "calibrated 2026-07 from network statements; see the relevant calib/<domain>/data/*_provenance",
        }
    )

    p = tac.get(cc, {})
    m = modes.get(cc, {})
    pk = peaks.get(cc, {})
    comp_rows.append(
        {
            "country_code": cc,
            "track_tac_eur_train_km": "" if tac_v is None else round(tac_v, 2),
            "track_tac_src": tac_src,
            "track_tac_night_mode": m.get("night_mode", "none"),
            "track_tac_night_band_start": m.get("night_band_start", ""),
            "track_tac_night_band_end": m.get("night_band_end", ""),
            "track_tac_night_full_if_accommodation": m.get(
                "night_full_if_accommodation", "False"
            ),
            **{
                f"track_tac_{k}": (
                    "" if _f_eur(p.get(k)) is None else round(_f_eur(p.get(k)), 6)
                )
                for k in (
                    "b_day",
                    "b_night",
                    "gamma",
                    "seat_km",
                    "revenue_share",
                    "fixed_per_train_km",
                    "per_stop",
                    "peak_multiplier",
                    "congestion_surcharge_eur_km",
                )
            },
            "track_tac_peak_band1_start": pk.get("band1_start", ""),
            "track_tac_peak_band1_end": pk.get("band1_end", ""),
            "track_tac_peak_band2_start": pk.get("band2_start", ""),
            "track_tac_peak_band2_end": pk.get("band2_end", ""),
            "track_tac_peak_weekdays_only": pk.get("weekdays_only", "False"),
        }
    )

write_seed("track_infrastructures.csv", TI_FIELDS, ti_rows)
write_seed("track_tac.csv", TAC_FIELDS, comp_rows)

## passage_charges

Crossing charges (EUR, one row per `passage_id`) joined with the crossing
polygon from `data/passage_geometries.geojson` as a GeoJSON string —
seed.py inserts them via `ST_GeomFromGeoJSON`. OERESUND_DK / OERESUND_SE
share the single OERESUND polygon: one crossing, two billing IMs.


In [ ]:
import json as _json

with open(
    CALIB_DIR / "tac" / "data" / "passage_geometries.geojson", encoding="utf-8"
) as fh:
    _geo = _json.load(fh)
GEOMETRY_BY_CROSSING = {
    f["properties"]["crossing_id"]: f["geometry"] for f in _geo["features"]
}

PASSAGE_NAMES = {
    "CHANNEL_TUNNEL": "Channel Tunnel",
    "STOREBAELT": "Storeb\u00e6lt (Great Belt) crossing",
    "OERESUND_DK": "\u00d8resund crossing \u2014 Danish half",
    "OERESUND_SE": "\u00d8resund crossing \u2014 Swedish half",
}

PC_FIELDS = [
    "passage_id",
    "passage_name",
    "passage_fixed_eur",
    "passage_per_passenger_eur",
    "passage_src",
    "passage_geom",
]

_by_id: dict[str, dict] = {}
for r in read_domain_data("tac", "passage_charges.csv"):
    pid = r["crossing_id"]
    fx = FX_TO_EUR.get(r["currency"] or "EUR", 1.0)
    row = _by_id.setdefault(
        pid,
        {
            "passage_id": pid,
            "passage_name": PASSAGE_NAMES[pid],
            "passage_fixed_eur": 0.0,
            "passage_per_passenger_eur": 0.0,
            "passage_src": r["source_id"],
            "passage_geom": _json.dumps(
                GEOMETRY_BY_CROSSING["OERESUND" if pid.startswith("OERESUND") else pid]
            ),
        },
    )
    if r["parameter"] == "fixed_per_train":
        row["passage_fixed_eur"] = round(float(r["value"]) * fx, 2)
    elif r["parameter"] == "per_passenger":
        row["passage_per_passenger_eur"] = round(float(r["value"]) * fx, 2)

pc_rows = [_by_id[pid] for pid in sorted(_by_id)]
write_seed("passage_charges.csv", PC_FIELDS, pc_rows)
for r in pc_rows:
    print(
        f"  {r['passage_id']:15} fixed {r['passage_fixed_eur']:>9} EUR"
        f"  per-pax {r['passage_per_passenger_eur']} EUR"
    )

## track_infrastructure_defaults

The fallback row for a country with no observation of its own. Median rather
than mean throughout: the default stands in for a country we know nothing
about, and a handful of very high-charging networks would drag a mean well
above anything typical.

In [ ]:
def _median(vals: list[float]):
    s = sorted(vals)
    n = len(s)
    if not n:
        return None
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2.0


def _col(rows, key):
    return [float(r[key]) for r in rows if r[key] not in ("", None)]


DEF_FIELDS = ["track_infra_default_key"] + [f for f in TI_FIELDS if f != "country_code"]
defaults = {
    "track_infra_default_key": "EU-MEDIAN-2032",
    "track_tac_eur_train_km": round(
        _median(_col(ti_rows, "track_tac_eur_train_km")), 2
    ),
    "track_tac_src": "IRG-TAC-2025",
    "track_parking_eur_day": round(_median(_col(ti_rows, "track_parking_eur_day")), 2),
    "track_parking_src": "",
    "track_shunting_eur_event": round(
        _median(_col(ti_rows, "track_shunting_eur_event")), 2
    ),
    "track_shunting_src": "NOX-MODEL",
    "track_energy_price_eur_kwh": round(
        _median(_col(ti_rows, "track_energy_price_eur_kwh")), 3
    ),
    "track_energy_price_src": "EUROSTAT-NRG-PC-205-C",
    "track_terrain_category": "Hilly",
    "track_terrain_score": round(_median(_col(ti_rows, "track_terrain_score"))),
    "track_terrain_src": "",
    "track_hsr_allowed": False,
    "track_hsr_src": "",
    "track_min_boarding_time": "00:02:00",
    "track_min_boarding_src": "",
    "track_min_alighting_time": "00:02:00",
    "track_min_alighting_src": "",
    "track_buffer_quota_per": round(
        _median(_col(ti_rows, "track_buffer_quota_per")), 4
    ),
    "track_buffer_src": "RMMS-9",
    "change_log": "EU medians over the calibrated countries",
}
write_seed("track_infrastructure_defaults.csv", DEF_FIELDS, [defaults])

# Referential integrity: every _src must resolve in sources.csv.
keys = {r["source_key"] for r in src_rows}
for row in ti_rows + [defaults]:
    for k, v in row.items():
        if k.endswith("_src") and v:
            assert v in keys, (
                f"{row.get('country_code', 'default')}/{k}: unknown source {v}"
            )
print("source key integrity OK")
print({k: v for k, v in defaults.items() if not k.endswith("_src")})